# Episode 4 — Control Flow with JIT

**Instructor notebook** · run top-to-bottom before recording.

Python `if` / `for` / `while` work eagerly, but **`jit` traces one path** through your control-flow graph. When the path depends on **input values**, you need `static_argnames`, `lax.cond`, or `lax.*_loop`.

| | |
|---|---|
| **Chapter** | 1.4 · Part I — Pure JAX |
| **Prereq** | Episodes 1–3 |
| **Next** | Episode 5 — pytrees and SGD |

**Source:** [Control flow and logical operators with JIT](https://docs.jax.dev/en/latest/control-flow.html)

In [58]:
from functools import partial

import jax
import jax.numpy as jnp
from jax import grad, jit, lax


## Control flow under `jit`

Eagerly, JAX behaves like NumPy. Under **`jit`**, Python control flow is evaluated at **compile time** — the compiled function is **one path** through the graph. Logical operators short-circuit the same way.

If the path depends on **input values**, tracing fails by default. It may depend on **shape/dtype** — then JAX **recompiles** on new shapes.

### What works

In [62]:
import time

def time_once(f, *args):
    start = time.perf_counter()
    f(*args)
    end = time.perf_counter()
    diff = end - start
    return print(f"Time taken: {diff:.2f} seconds")


@jit
def f(x):
    # Tracer<Int32>, .dtype, .shape
    for i in range(3):  # compile-time constant — always 3 trips
        x = 2 * x
    return x


time_once(f, 3) # TRACE + COMPILE + EXECUTE
"""
GLOBAL JIT CACHE
key = (f, (x: int<32>), value = compiled function
"""

time_once(f, 4)




Time taken: 0.03 seconds
Time taken: 0.00 seconds


In [65]:
@jit
def g(x):
    # Tracer<float32, 3>, .dtype, .shape
    y = 0.0
    for i in range(x.shape[0]):  # trip count from shape, not value
        y = y + x[i]
    return y

time_once(g, jnp.array([1.0, 2.0, 3.0])) # TRACE + COMPILE + EXECUTE
time_once(g, jnp.array([4.0, 5.0, 6.0]))

Time taken: 0.04 seconds
Time taken: 0.00 seconds


### What fails (value-dependent branch)

In [66]:
@jit
def f(x):
    # Tracer<Int32>
    if x < 3:
        return 3.0 * x ** 2
    else:
        return -4 * x


try:
    f(2)
except jax.errors.TracerBoolConversionError as e:
    print("TracerBoolConversionError:", e)


TracerBoolConversionError: Attempted boolean conversion of traced array with shape bool[].
The error occurred while tracing the function f at /tmp/ipykernel_365059/3487195452.py:1 for jit. This concrete value was not available in Python because it depends on the value of the argument x.
See https://docs.jax.dev/en/latest/errors.html#jax.errors.TracerBoolConversionError


In [67]:
@jit
def g(x):
    # Tracer<Int32> (.dtype, .shape)
    return (x > 0) and (x < 3)


try:
    g(2)
except jax.errors.TracerBoolConversionError as e:
    print("TracerBoolConversionError:", e)


TracerBoolConversionError: Attempted boolean conversion of traced array with shape bool[].
The error occurred while tracing the function g at /tmp/ipykernel_365059/4181119907.py:1 for jit. This concrete value was not available in Python because it depends on the value of the argument x.
See https://docs.jax.dev/en/latest/errors.html#jax.errors.TracerBoolConversionError


### Why?

`jit` traces on **`ShapedArray`** — shape + dtype, not a specific value. For `if x < 3`, the predicate is `{True, False}` abstractly; Python cannot pick a branch, so tracing stops.

**Tradeoff:** more abstract traces → fewer recompiles, but stricter rules on Python control flow.

### Branches on `.shape` vs `static_argnames`

Under `jit`, JAX traces on **`ShapedArray`** — shape and dtype are concrete. Branch on `.shape` and JAX compiles **one path per shape** (recompiles when shape changes).

For branches on a **Python scalar's value** (not shape), use `static_argnames` (below).

In [ ]:
from jax import make_jaxpr


def f(x):
    if x.shape[0] < 3:  # shape is concrete at trace time
        return 3.0 * jnp.sum(x ** 2)
    else:
        return -4.0 * jnp.sum(x)


print("shape (2,):")
print(make_jaxpr(f)(jnp.array([1.0, 2.0]))) # compiled binary only 1 path

print("shape (4,):")
print(make_jaxpr(f)(jnp.array([1.0, 2.0, 3.0, 4.0])))


shape (2,):
{ lambda ; a:f32[2]. let
    b:f32[2] = integer_pow[y=2] a
    c:f32[] = reduce_sum[axes=(0,)] b
    d:f32[] = mul 3.0:f32[] c
  in (d,) }
shape (4,):
{ lambda ; a:f32[4]. let
    b:f32[] = reduce_sum[axes=(0,)] a
    c:f32[] = mul -4.0:f32[] b
  in (c,) }


In [73]:
def sum_first_n(x, n):
    # x: Tracer(float32[3]), 
    # n: int<32> WHICH HAS A SHAPE, DTYPE + VALUE
    y = 0.0
    for i in range(n):
        y = y + x[i]


    return y


sum_first_n_jit = jit(sum_first_n, static_argnames="n")


# print(sum_first_n_jit(jnp.array([2.0, 3.0, 4.0]), 2))
time_once(sum_first_n_jit, jnp.array([2.0, 3.0, 4.0]), 2)
time_once(sum_first_n_jit, jnp.array([2.0, 3.0, 4.0]), 3)
time_once(sum_first_n_jit, jnp.array([2.0, 3.0, 4.0]), 2)



Time taken: 0.04 seconds
Time taken: 0.04 seconds
Time taken: 0.00 seconds


With `static_argnames='n'`, the loop is **statically unrolled** at trace time (fine for small `n`; disastrous if `n` changes every call).

⚠️ **`static_argnames` can be handy when `length` rarely changes, but costly if it changes a lot.**

### Value-dependent shapes

Same issue when **array shapes** depend on argument **values** (shape-specialization on dtype/shape alone is OK):

In [75]:
def example_fun(length, val):
    # length: Tracer(int<32>), val: Tracer(float32)
    return jnp.ones((length,)) * val


print(example_fun(5, 4))


[4. 4. 4. 4. 4.]


In [76]:
bad_example_jit = jit(example_fun)

try:
    bad_example_jit(10, 4)
except TypeError as e:
    print("TypeError:", e)


TypeError: Shapes must be 1D sequences of concrete values of integer type, got (Traced<~int32[]>with<DynamicJaxprTrace>,).
If using `jit`, try using `static_argnums` or applying `jit` to smaller subfunctions.
The error occurred while tracing the function example_fun at /tmp/ipykernel_365059/1962443222.py:1 for jit. This concrete value was not available in Python because it depends on the value of the argument length.


In [77]:
good_example_jit = jit(example_fun, static_argnames="length")
print(good_example_jit(10, 4))
print(good_example_jit(5, 4))  # recompiles — new output shape


[4. 4. 4. 4. 4. 4. 4. 4. 4. 4.]
[4. 4. 4. 4. 4.]


### Side effects inside `jit`

`print` runs at trace time and shows **tracers**, not concrete values. For debug output inside compiled code, use [`jax.debug.print`](https://docs.jax.dev/en/latest/_autosummary/jax.debug.print.html) (Episode 2).

In [78]:
@jit
def f(x):
    print("x:", x)
    y = 2 * x
    print("y:", y)
    return y


f(2)


x: Traced<~int32[]>with<DynamicJaxprTrace>
y: Traced<~int32[]>with<DynamicJaxprTrace>


Array(4, dtype=int32, weak_type=True)

## Structured control flow primitives

When you want **traceable** control flow **without** recompiling on every branch — and **without** unrolling large loops — use `jax.lax`:

| Primitive | Role |
|---|---|
| `lax.cond` | differentiable `if` on a scalar predicate |
| `lax.while_loop` | `while` with runtime stop condition |
| `lax.fori_loop` | counted `for`; XLA may lower to `scan` or `while_loop` |
| `lax.scan` | fold/map/scan with per-step inputs |

### `lax.cond`

In [79]:
operand = jnp.array([0.0])

def f(x):
    return x + 1

def g(x):
    return x - 1

print(lax.cond(True, f, g, operand)) # true --> apply arg1 (f) --> 1
print(lax.cond(False, f, g, operand)) # false --> apply arg2 (g) --> -1


[1.]
[-1.]


Related: [`lax.select`](https://docs.jax.dev/en/latest/_autosummary/jax.lax.select.html) (batched choices as arrays), [`lax.switch`](https://docs.jax.dev/en/latest/_autosummary/jax.lax.switch.html) (multi-branch). NumPy-style: `jnp.where`, `jnp.piecewise`, `jnp.select`.

### `lax.while_loop`

In [81]:
init_val = (-5, 0) # (value, trip_count)

def cond_fun(x):
    return x[0] < 10

def body_fun(x):
    return x[0] + 2, x[1] + 1

lax.while_loop(cond_fun, body_fun, init_val)

(Array(11, dtype=int32, weak_type=True), Array(8, dtype=int32, weak_type=True))

### `lax.fori_loop`

In [82]:
init_val = 0
def body_fun(i, x):
    return x + i

print(lax.fori_loop(0, 10, body_fun, init_val)) # can also scan


45


### Summary — `jit` vs `grad`

| construct | `jit` | `grad` |
|---|---|---|
| `if` | ❌ (value-dependent) | ✔ |
| `for` | ✔* | ✔ |
| `while` | ✔* | ✔ |
| `lax.cond` | ✔ | ✔ |
| `lax.while_loop` | ✔ | fwd only |
| `lax.fori_loop` | ✔ | fwd only† |
| `lax.scan` | ✔ | ✔ |

\* loop bound must be **value-independent** (compile-time / shape-based) — otherwise unrolls or fails.

† `fori_loop` is rev-mode differentiable when endpoints are static.

## Logical operators

Use **`jnp.logical_and` / `logical_or` / `logical_not`** (or bitwise `&` `|` `~`) under `jit`. Unlike Python `and`/`or`, they **do not short-circuit** — both sides are evaluated.

In [83]:
def python_check_positive_even(x):
    is_even = x % 2 == 0
    return is_even and (x > 0)


@jit
def jax_check_positive_even(x):
    is_even = x % 2 == 0
    return jnp.logical_and(is_even, x > 0)


print(python_check_positive_even(24))
print(jax_check_positive_even(24))


True
True


In [84]:
x = jnp.array([-1, 2, 5])
print(jax_check_positive_even(x))


[False  True False]


In [85]:
try:
    python_check_positive_even(x)
except ValueError as e:
    print("ValueError:", e)


ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()


## Python control flow + `grad` (no `jit`)

The constraints above apply to **`jit` only**. **`grad`** on pure Python functions with `if`/`for` works fine — Autograd-style:

In [86]:
def f(x):
    if x < 3:
        return 3.0 * x ** 2
    else:
        return -4 * x


print(grad(f)(2.0))
print(grad(f)(4.0))


12.0
-4.0


> **Key insight:** Under `jit`, value-dependent Python control flow fails or forces recompilation via `static_argnames`. For dynamic branches and loops inside one compiled graph, use **`lax.cond`**, **`lax.while_loop`**, **`lax.fori_loop`**, or **`lax.scan`**.

---
## Exercise

1. Write a `@jit` function with `if x < 0` that fails; fix it with `static_argnames` or `lax.cond`.
2. Write `example_fun(length, val)` with `jit` and `static_argnames='length'`; call it with two different lengths.
3. Implement a counted sum with `lax.fori_loop` and a runtime stop with `lax.while_loop`.
4. Replace `(x > 0) and (x < 3)` with `jnp.logical_and` under `jit`.

*(Solution below.)*

In [ ]:
@jit
def abs_scaled(x):
    return lax.cond(x < 0, lambda v: -2.0 * v, lambda v: 2.0 * v, x)


print("cond:", abs_scaled(-3.0), abs_scaled(3.0))
print("shapes:", good_example_jit(3, 1.0).shape, good_example_jit(7, 1.0).shape)
print("fori:", lax.fori_loop(0, 5, lambda i, a: a + i, 0))
print("while:", lax.while_loop(lambda s: s < 5, lambda s: s + 1, 0))
print("logical:", jax_check_positive_even(jnp.array([2, 4, -2])))


**Next:** Episode 5 — pytrees and SGD.